In [4]:

# ============================================================
# CASE STUDY 1: ARITHMETIC EXPRESSION VALIDATION
# Course: Formal Language and Automata (21CSC301T)
# Technique: Context-Free Grammar and Predictive PDA
# ============================================================

import re

# 1. CONTEXT-FREE GRAMMAR

grammar = {
    "E": [["T", "E'"]],
    "E'": [["+", "T", "E'"], ["-", "T", "E'"], ["ε"]],
    "T": [["F", "T'"]],
    "T'": [["*", "F", "T'"], ["/", "F", "T'"], ["ε"]],
    "F": [["(", "E", ")"], ["id"]]
}

print("=" * 60)
print("CASE STUDY 1: ARITHMETIC EXPRESSION VALIDATION")
print("=" * 60)

print("\nCONTEXT-FREE GRAMMAR:")
for non_terminal, productions in grammar.items():
    rhs = [" ".join(p) for p in productions]
    print(f"{non_terminal} -> {' | '.join(rhs)}")


# 2. TOKENIZATION

def tokenize(expression):
    pattern = r'\d+(?:\.\d+)?|[A-Za-z_]\w*|[()+*/-]'
    tokens = re.findall(pattern, expression)

    if ''.join(tokens) != re.sub(r'\s+', '', expression):
        raise ValueError("Invalid character in expression")

    return [
        "id" if re.fullmatch(
            r'\d+(?:\.\d+)?|[A-Za-z_]\w*', token
        ) else token
        for token in tokens
    ]


# 3. PREDICTIVE PDA SIMULATION

def arithmetic_pda(expression, show_trace=True):

    try:
        tokens = tokenize(expression)
    except ValueError:
        if show_trace:
            print("\nInput:", expression)
            print("Result: REJECTED")
        return False

    tokens.append("$")
    stack = ["$", "E"]
    index = 0

    terminals = {"+", "-", "*", "/", "(", ")", "id", "$"}

    trace = []
    accepted = False

    for step in range(200):

        if not stack:
            break

        top = stack.pop()
        current = tokens[index]

        trace.append(
            (step + 1, top, current, stack.copy())
        )

        # Terminal matching
        if top in terminals:

            if top == current:
                index += 1
            else:
                break

            if top == "$":
                accepted = (not stack and index == len(tokens))
                break

        # Non-terminal expansion
        else:

            production = None

            if top == "E":
                if current in {"id", "("}:
                    production = ["T", "E'"]

            elif top == "E'":
                if current in {"+", "-"}:
                    production = [current, "T", "E'"]
                elif current in {")", "$"}:
                    production = ["ε"]

            elif top == "T":
                if current in {"id", "("}:
                    production = ["F", "T'"]

            elif top == "T'":
                if current in {"*", "/"}:
                    production = [current, "F", "T'"]
                elif current in {"+", "-", ")", "$"}:
                    production = ["ε"]

            elif top == "F":
                if current == "(":
                    production = ["(", "E", ")"]
                elif current == "id":
                    production = ["id"]

            if production is None:
                break

            if production != ["ε"]:
                stack.extend(reversed(production))

    if show_trace:
        print("\nINPUT:", expression)
        print("\nPDA EXECUTION TRACE")
        print("-" * 60)
        print(f"{'Step':<8}{'Top':<12}{'Input':<12}{'Stack'}")

        for step, top, current, remaining in trace:
            print(
                f"{step:<8}{top:<12}{current:<12}{remaining}"
            )

        print("-" * 60)
        print("FINAL RESULT:",
              "ACCEPTED" if accepted else "REJECTED")

    return accepted


# 4. TEST CASES

test_cases = [
    ("a+b", True),
    ("(a+b)*c", True),
    ("12+5*3", True),
    ("a*(b+c)", True),
    ("(a+b", False),
    ("a++b", False),
    ("*a+b", False),
    ("a+b)", False)
]

print("\n\nTEST CASE VALIDATION")
print("-" * 60)

correct = 0

for expression, expected in test_cases:

    result = arithmetic_pda(expression, show_trace=False)

    status = "ACCEPTED" if result else "REJECTED"
    expected_status = "ACCEPTED" if expected else "REJECTED"

    match = result == expected

    if match:
        correct += 1

    print(
        f"{expression:<15} {status:<12} "
        f"Expected: {expected_status:<10} "
        f"{'PASS' if match else 'FAIL'}"
    )

print("-" * 60)
print(f"Total Tests: {len(test_cases)}")
print(f"Passed: {correct}")
print(f"Failed: {len(test_cases) - correct}")


# 5. DETAILED PDA EXECUTION

print("\n\nDETAILED PDA EXECUTION")
arithmetic_pda("(a+b)*c", show_trace=True)


# 6. FINAL SUMMARY

print("\n" + "=" * 60)
print("CASE STUDY 1 COMPLETED")
print("Method: CFG-based Predictive PDA")
print("Validation: Arithmetic Expression Syntax")
print("Operator Precedence: *, / before +, -")
print("=" * 60)


CASE STUDY 1: ARITHMETIC EXPRESSION VALIDATION

CONTEXT-FREE GRAMMAR:
E -> T E'
E' -> + T E' | - T E' | ε
T -> F T'
T' -> * F T' | / F T' | ε
F -> ( E ) | id


TEST CASE VALIDATION
------------------------------------------------------------
a+b             ACCEPTED     Expected: ACCEPTED   PASS
(a+b)*c         ACCEPTED     Expected: ACCEPTED   PASS
12+5*3          ACCEPTED     Expected: ACCEPTED   PASS
a*(b+c)         ACCEPTED     Expected: ACCEPTED   PASS
(a+b            REJECTED     Expected: REJECTED   PASS
a++b            REJECTED     Expected: REJECTED   PASS
*a+b            REJECTED     Expected: REJECTED   PASS
a+b)            REJECTED     Expected: REJECTED   PASS
------------------------------------------------------------
Total Tests: 8
Passed: 8
Failed: 0


DETAILED PDA EXECUTION

INPUT: (a+b)*c

PDA EXECUTION TRACE
------------------------------------------------------------
Step    Top         Input       Stack
1       E           (           ['$']
2       T           (  

In [5]:

# ============================================================
# CASE STUDY 2: BALANCED PARENTHESES CHECKER
# ============================================================

# Context-Free Grammar:
# S -> (S)S | ε
#
# PDA Method:
# Push '(' onto the stack when an opening parenthesis is found.
# Pop '(' when a closing parenthesis is found.
# Reject if a closing parenthesis has no matching opening one.
# Accept if the stack is empty after processing the input.

def balanced_parentheses(expression, show_trace=False):

    stack = []
    trace = []

    for position, symbol in enumerate(expression, start=1):

        if symbol == '(':
            stack.append('(')
            action = "PUSH '('"

        elif symbol == ')':
            if not stack:
                trace.append((position, symbol, "REJECT", stack.copy()))
                if show_trace:
                    print_trace(trace)
                return "REJECTED"

            stack.pop()
            action = "POP '('"

        else:
            trace.append((position, symbol, "INVALID SYMBOL", stack.copy()))
            if show_trace:
                print_trace(trace)
            return "REJECTED"

        trace.append((position, symbol, action, stack.copy()))

    result = "ACCEPTED" if not stack else "REJECTED"

    if show_trace:
        print_trace(trace)
        print("\nFINAL STACK:", stack)
        print("FINAL RESULT:", result)

    return result


def print_trace(trace):
    print("\nPDA EXECUTION TRACE")
    print("-" * 65)
    print(f"{'Step':<8}{'Symbol':<12}{'Action':<20}{'Stack'}")
    print("-" * 65)

    for step, symbol, action, stack in trace:
        print(f"{step:<8}{symbol:<12}{action:<20}{stack}")


# ---------------- TEST CASE VALIDATION ----------------

test_cases = [
    ("()", "ACCEPTED"),
    ("(())", "ACCEPTED"),
    ("()()", "ACCEPTED"),
    ("((()))", "ACCEPTED"),
    ("(()", "REJECTED"),
    ("())", "REJECTED"),
    (")(", "REJECTED"),
    ("(()))", "REJECTED")
]

print("=" * 65)
print("CASE STUDY 2: BALANCED PARENTHESES CHECKER")
print("=" * 65)

print("\nCONTEXT-FREE GRAMMAR:")
print("S -> (S)S | ε")

print("\nTEST CASE VALIDATION")
print("-" * 65)
print(f"{'Input':<15}{'Actual Result':<18}{'Expected':<15}{'Status'}")
print("-" * 65)

passed = 0
failed = 0

for expression, expected in test_cases:
    actual = balanced_parentheses(expression)
    status = "PASS" if actual == expected else "FAIL"

    print(f"{expression:<15}{actual:<18}{expected:<15}{status}")

    if status == "PASS":
        passed += 1
    else:
        failed += 1

print("-" * 65)
print("Total Tests:", len(test_cases))
print("Passed:", passed)
print("Failed:", failed)


# ---------------- DETAILED PDA EXECUTION ----------------

print("\n\nDETAILED PDA EXECUTION")
print("-" * 65)
print("INPUT: ((()))")

balanced_parentheses("((()))", show_trace=True)

print("\n" + "=" * 65)
print("CASE STUDY 2 COMPLETED")
print("Method: Deterministic Pushdown Automaton (DPDA)")
print("Validation: Balanced Parentheses")
print("Stack Operation: Push '(' and Pop ')'")
print("=" * 65)


CASE STUDY 2: BALANCED PARENTHESES CHECKER

CONTEXT-FREE GRAMMAR:
S -> (S)S | ε

TEST CASE VALIDATION
-----------------------------------------------------------------
Input          Actual Result     Expected       Status
-----------------------------------------------------------------
()             ACCEPTED          ACCEPTED       PASS
(())           ACCEPTED          ACCEPTED       PASS
()()           ACCEPTED          ACCEPTED       PASS
((()))         ACCEPTED          ACCEPTED       PASS
(()            REJECTED          REJECTED       PASS
())            REJECTED          REJECTED       PASS
)(             REJECTED          REJECTED       PASS
(()))          REJECTED          REJECTED       PASS
-----------------------------------------------------------------
Total Tests: 8
Passed: 8
Failed: 0


DETAILED PDA EXECUTION
-----------------------------------------------------------------
INPUT: ((()))

PDA EXECUTION TRACE
------------------------------------------------------------

In [6]:

# ============================================================
# CASE STUDY 3: PALINDROME CHECKER
# ============================================================

# Context-Free Grammar:
# S -> aSa | bSb | a | b | ε
#
# Language: Palindromes over {a, b}
#
# NPDA Method:
# 1. Push the first half of the input onto the stack.
# 2. Nondeterministically guess the middle.
# 3. For odd-length strings, skip the middle symbol.
# 4. Match the remaining input with the stack by popping.
# 5. Accept if the input is consumed and the stack is empty.

from collections import deque

def palindrome_npda(string, show_trace=False):

    if any(ch not in "ab" for ch in string):
        return "REJECTED"

    # Configuration: (state, input_position, stack)
    start = ("PUSH", 0, ())
    queue = deque([start])

    visited = {start}
    parent = {start: None}
    action_used = {}

    accepted_config = None

    while queue:
        state, pos, stack = queue.popleft()

        # Accept when all input is consumed and stack is empty
        if pos == len(string) and len(stack) == 0 and state == "MATCH":
            accepted_config = (state, pos, stack)
            break

        next_configs = []

        if state == "PUSH":

            # Push the next input symbol onto the stack
            if pos < len(string):
                symbol = string[pos]
                next_configs.append((
                    ("PUSH", pos + 1, stack + (symbol,)),
                    f"PUSH '{symbol}'"
                ))

            # Guess an even-length midpoint
            next_configs.append((
                ("MATCH", pos, stack),
                "GUESS EVEN MIDPOINT"
            ))

            # Guess an odd-length midpoint
            if pos < len(string):
                next_configs.append((
                    ("MATCH", pos + 1, stack),
                    f"SKIP MIDDLE '{string[pos]}'"
                ))

        elif state == "MATCH":

            # Match current input symbol with stack top
            if pos < len(string) and stack:
                if string[pos] == stack[-1]:
                    next_configs.append((
                        ("MATCH", pos + 1, stack[:-1]),
                        f"MATCH AND POP '{string[pos]}'"
                    ))

        for config, action in next_configs:
            if config not in visited:
                visited.add(config)
                parent[config] = (state, pos, stack)
                action_used[config] = action
                queue.append(config)

    if accepted_config is None:
        return "REJECTED"

    # Reconstruct successful execution path
    path = []
    current = accepted_config

    while current is not None:
        path.append(current)
        current = parent[current]

    path.reverse()

    if show_trace:
        print("\nNPDA EXECUTION TRACE")
        print("-" * 85)
        print(f"{'Step':<8}{'State':<12}{'Input Position':<18}{'Stack':<18}{'Action'}")
        print("-" * 85)

        for i, config in enumerate(path):
            state, pos, stack = config

            if i == 0:
                action = "START"
            else:
                action = action_used[config]

            print(f"{i + 1:<8}{state:<12}{pos:<18}{str(list(stack)):<18}{action}")

        print("-" * 85)
        print("FINAL STACK:", list(accepted_config[2]))
        print("FINAL RESULT: ACCEPTED")

    return "ACCEPTED"


# ---------------- TEST CASE VALIDATION ----------------

test_cases = [
    ("a", "ACCEPTED"),
    ("aa", "ACCEPTED"),
    ("aba", "ACCEPTED"),
    ("abba", "ACCEPTED"),
    ("baab", "ACCEPTED"),
    ("", "ACCEPTED"),
    ("aab", "REJECTED"),
    ("abb", "REJECTED"),
    ("abab", "REJECTED"),
    ("abc", "REJECTED")
]

print("=" * 75)
print("CASE STUDY 3: PALINDROME CHECKER")
print("=" * 75)

print("\nCONTEXT-FREE GRAMMAR:")
print("S -> aSa | bSb | a | b | ε")

print("\nTEST CASE VALIDATION")
print("-" * 75)
print(f"{'Input':<15}{'Actual Result':<18}{'Expected':<15}{'Status'}")
print("-" * 75)

passed = 0
failed = 0

for string, expected in test_cases:
    actual = palindrome_npda(string)
    display_input = string if string else "ε"
    status = "PASS" if actual == expected else "FAIL"

    print(f"{display_input:<15}{actual:<18}{expected:<15}{status}")

    if status == "PASS":
        passed += 1
    else:
        failed += 1

print("-" * 75)
print("Total Tests:", len(test_cases))
print("Passed:", passed)
print("Failed:", failed)


# ---------------- DETAILED NPDA EXECUTION ----------------

print("\n\nDETAILED NPDA EXECUTION")
print("-" * 75)
print("INPUT: abba")

palindrome_npda("abba", show_trace=True)

print("\n" + "=" * 75)
print("CASE STUDY 3 COMPLETED")
print("Method: Non-Deterministic Pushdown Automaton (NPDA)")
print("Validation: Palindrome Strings over {a, b}")
print("Stack Operation: Push First Half, Match and Pop Second Half")
print("=" * 75)


CASE STUDY 3: PALINDROME CHECKER

CONTEXT-FREE GRAMMAR:
S -> aSa | bSb | a | b | ε

TEST CASE VALIDATION
---------------------------------------------------------------------------
Input          Actual Result     Expected       Status
---------------------------------------------------------------------------
a              ACCEPTED          ACCEPTED       PASS
aa             ACCEPTED          ACCEPTED       PASS
aba            ACCEPTED          ACCEPTED       PASS
abba           ACCEPTED          ACCEPTED       PASS
baab           ACCEPTED          ACCEPTED       PASS
ε              ACCEPTED          ACCEPTED       PASS
aab            REJECTED          REJECTED       PASS
abb            REJECTED          REJECTED       PASS
abab           REJECTED          REJECTED       PASS
abc            REJECTED          REJECTED       PASS
---------------------------------------------------------------------------
Total Tests: 10
Passed: 10
Failed: 0


DETAILED NPDA EXECUTION
-------------------